# Ipotesi 1 — 00. Preparazione della condizione C3 (video scambiato)

Gira in locale, senza GPU. Produce i due file che servono ai notebook dei modelli:

| File | Contenuto |
|---|---|
| `dati/swap_map.json` | per ogni domanda: video originale, video di scambio, ricette di entrambi |
| `dati/annotazioni_swap/nutrition_video_nutrition_estimation.json` | le annotazioni del benchmark con **solo** l'id del video sostituito |

**Regole dello scambio.** Il video di scambio:
1. e' uno dei 14 video gia' usati da `video_nutrition_estimation` (gia' su Drive, niente download);
2. mostra una ricetta **diversa** da quella della domanda (il partecipante puo' essere diverso);
3. non contiene **nessuna** delle 5 opzioni fra i suoi ingredienti.

Con la regola 3 la condizione e' netta: nel video scambiato nessuna risposta e' visibile,
quindi un modello che guarda il video non ha nulla a sostegno della risposta che darebbe col video vero.

Le ricette complete vengono da `high-level/complete_recipes.json` del repository HD-EPIC,
copiato in `dati/`.

In [ ]:
import json, sys
from collections import Counter
from pathlib import Path

RADICE = Path.cwd().resolve()
if RADICE.name == "notebook":
    RADICE = RADICE.parent
PROGETTO = RADICE.parent
sys.path.insert(0, str(RADICE / "codice"))

import hdepic_eval as H
import ablazione as A

ANN_FILE = PROGETTO / "preprocessing_folder" / A.TASK_FILE
RICETTE  = RADICE / "dati" / "complete_recipes.json"
DATI     = RADICE / "dati"
SEED = 0

ann = json.loads(ANN_FILE.read_text())
info = A.info_video(json.loads(RICETTE.read_text()))
pool = sorted({x["inputs"]["video 1"]["id"] for x in ann.values()})
print(f"{len(ann)} domande, {len(pool)} video, {len(info)} video con ricetta nota")
assert all(v in info for v in pool), "un video del benchmark non ha ricetta"

50 domande, 14 video, 116 video con ricetta nota


## 1. Quanti video di scambio validi ha ogni domanda?

In [2]:
n_cand = {k: len(A.candidati_scambio(x, x["inputs"]["video 1"]["id"], pool, info)) for k, x in ann.items()}
print("candidati per domanda: min", min(n_cand.values()), "| max", max(n_cand.values()))
print("distribuzione:", dict(sorted(Counter(n_cand.values()).items())))

candidati per domanda: min 3 | max 13
distribuzione: {3: 7, 4: 6, 5: 12, 6: 3, 7: 10, 8: 4, 9: 5, 10: 1, 12: 1, 13: 1}


## 2. Assegnazione degli scambi

Fra i candidati si sceglie il video finora usato meno volte (pareggi rotti con seed fisso),
cosi' nessun video di scambio pesa troppo sul risultato. La mappa e' la stessa per i due modelli.

In [3]:
swap_map = A.costruisci_swap_map(ann, info, pool, seed=SEED)
A.verifica_swap_map(ann, swap_map, info)
print("vincoli verificati su tutte le domande\n")

import pandas as pd
vista = pd.DataFrame.from_dict(swap_map, orient="index")
vista.index = [k.split("_")[-1] for k in vista.index]
print(vista[["ricetta_originale", "ricetta_scambio", "stesso_partecipante"]].to_string())

vincoli verificati su tutte le domande

                                                                          ricetta_originale                                                                          ricetta_scambio  stesso_partecipante
0                                                                  Red Lentils Squash Curry                                                  Salmon Teriyaki Skewers with Vegetables                False
1                                                                           Beef Stroganoff                                                                           Thai Red Curry                False
2                                                                  Red Lentils Squash Curry                                                                             Aloo Paratha                False
3                                                   Salmon Teriyaki Skewers with Vegetables                                                             

In [4]:
uso = Counter(s["video_scambio"] for s in swap_map.values())
print("uso dei video di scambio:", dict(sorted(uso.items())))
print("stesso partecipante:", sum(s["stesso_partecipante"] for s in swap_map.values()), "/", len(swap_map))

uso dei video di scambio: {'P03-20240217-131219': 3, 'P03-20240218-190556': 4, 'P04-20240414-065311': 4, 'P06-20240510-100047': 4, 'P06-20240510-104642': 2, 'P06-20240510-140459': 4, 'P07-20240531-201703': 3, 'P08-20240613-123547': 4, 'P08-20240614-085000': 3, 'P08-20240614-184945': 4, 'P08-20240617-130401': 3, 'P08-20240617-184909': 3, 'P08-20240622-141902': 4, 'P09-20240621-093545': 5}
stesso partecipante: 6 / 50


## 3. Un esempio, per controllo a occhio

In [5]:
k = sorted(ann, key=lambda k: int(k.split("_")[-1]))[0]
s = swap_map[k]
print(ann[k]["question"])
for i, c in enumerate(ann[k]["choices"]):
    print(f"  {H.LETTERS[i]}. {c}{'   <- corretta' if i == ann[k]['correct_idx'] else ''}")
print(f"\nvideo vero    : {s['video_originale']}  ({s['ricetta_originale']})")
print(f"video scambio : {s['video_scambio']}  ({s['ricetta_scambio']})")
print("ingredienti del video scambiato:", sorted(info[s["video_scambio"]]["ingredienti"]))

What is the ingredient with highest carbs in this recipe?
  A. butternut squash
  B. onions
  C. spring greens
  D. split red lentils   <- corretta
  E. spicy garlic & coconut paste, ginger lentil stock and dhal spice mix

video vero    : P03-20240217-131219  (Red Lentils Squash Curry)
video scambio : P08-20240614-184945  (Salmon Teriyaki Skewers with Vegetables)
ingredienti del video scambiato: ['black pepper', 'broccoli', 'chickpea, spinach & quinoa mix', 'garlic powder', 'olive oil cooking spray', 'oregano', 'salt', 'teriyaki salmon kebabs']


## 4. Salvataggio

In [6]:
(DATI / "annotazioni_swap").mkdir(parents=True, exist_ok=True)
(DATI / "swap_map.json").write_text(json.dumps(swap_map, indent=1, ensure_ascii=False))
(DATI / "annotazioni_swap" / A.TASK_FILE).write_text(
    json.dumps(A.annotazioni_scambiate(ann, swap_map), indent=1, ensure_ascii=False))

A.controlla_annotazioni_scambiate(ANN_FILE.parent, DATI / "annotazioni_swap")
print("salvati:", DATI / "swap_map.json", "e", DATI / "annotazioni_swap" / A.TASK_FILE)

OK: 50 domande, cambia solo il video
salvati: /Users/lorenzodidomenico/Desktop/PROGETTO DEEP LEARNING 1/tesi_ipotesi1_video/dati/swap_map.json e /Users/lorenzodidomenico/Desktop/PROGETTO DEEP LEARNING 1/tesi_ipotesi1_video/dati/annotazioni_swap/nutrition_video_nutrition_estimation.json


**Passo successivo:** caricare la cartella `tesi_ipotesi1_video` su Drive, dentro
`Progetto Deep Learning 1/`, e aprire `01_videollama2.ipynb` su Colab.